In [1]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.8/923.8 kB 7.7 MB/s eta 0:00:00


In [7]:
import os
import anthropic
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
# Creating functions to run locally
def add_numbers(num1, num2):
    return num1 + num2

def multiply_numbers(num1, num2):
    return num1 * num2

def divide_numbers(num1, num2):
    if num2 == 0:
        return "Error: Division by zero is undefined."
    return num1 / num2

# Creating Schema
add_schema = {
    "name": "add_numbers",
    "description": "Adds two numbers. Use for addition or summing.",
    "input_schema": {
        "type": "object",
        "properties": {
            "num1": {"type": "number", "description": "First number"},
            "num2": {"type": "number", "description": "Second number"}
        },
        "required": ["num1", "num2"]
    }
}

multiply_schema = {
    "name": "multiply_numbers",
    "description": "Multiplies two numbers. Use for multiplication or products.",
    "input_schema": {
        "type": "object",
        "properties": {
            "num1": {"type": "number", "description": "First number"},
            "num2": {"type": "number", "description": "Second number"}
        },
        "required": ["num1", "num2"]
    }
}

divide_schema = {
    "name": "divide_numbers",
    "description": "Divides first number by second. Use for division.",
    "input_schema": {
        "type": "object",
        "properties": {
            "num1": {"type": "number", "description": "Dividend"},
            "num2": {"type": "number", "description": "Divisor (cannot be zero)"}
        },
        "required": ["num1", "num2"]
    }
}

# Combine Scema to a list
tools_list = [add_schema, multiply_schema, divide_schema]

# Routing Map
tool_routing_map = {
    "add_numbers":      add_numbers,
    "multiply_numbers": multiply_numbers,
    "divide_numbers":   divide_numbers,
}

def user_msg(content):
    return {"role": "user", "content": content}

def assistant_msg(content):
    return {"role": "assistant", "content": content}

# Calling Claude
client = anthropic.Anthropic()

# Creating Chat function
def chat(messages):
    return client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        tools=tools_list,
        messages=messages
    )

# Creating agent
def run_math_agent():
    user_prompt = input("Your question: ")
    print(f"\nUSER: {user_prompt}\n")

    # Start conversation history
    messages = [user_msg(user_prompt)]

    # Send to Claude
    print("Sending prompt and tool schemas to Claude...")
    response = chat(messages)

    # LOOP — keep running until Claude has no more tools to call
    while response.stop_reason == "tool_use":

        tool_calls = [b for b in response.content if b.type == "tool_use"]
        print(f"\n[PHASE B] Claude wants to call {len(tool_calls)} tool(s)")

        # Collect ALL tool results in this list
        tool_results = []

        for block in response.content:
            if block.type == "tool_use":
                chosen_name = block.name
                args        = block.input
                tool_use_id = block.id

                print(f"\n -> Tool : '{chosen_name}'")
                print(f" -> Args : {args}")

                # Run each tool locally
                result = tool_routing_map[chosen_name](args["num1"], args["num2"])
                print(f" -> Result: {result}")

                # Collect result with tracking id
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": str(result)
                })

        # Add assistant turn + all tool results to history
        messages.append(assistant_msg(response.content))
        messages.append(user_msg(tool_results))

        # Send updated history back to Claude
        print("\n[PHASE D] Sending all results back to Claude...")
        response = chat(messages)

    # Claude finished — print final answer
    print("\n==============================================")
    print("CLAUDE:", response.content[0].text)
    print("==============================================")

run_math_agent()

Your question: what is the difference between add and multiply?

USER: what is the difference between add and multiply?

[PHASE A] Sending prompt and tool schemas to Claude...

CLAUDE: The difference between **add** and **multiply** is:

1. **Add (Addition)**
   - Combines two numbers by putting them together
   - Example: 3 + 5 = 8
   - You're finding the total when combining quantities

2. **Multiply (Multiplication)**
   - Combines two numbers by repeating one number a certain number of times
   - Example: 3 × 5 = 15 (which is the same as 5 + 5 + 5, or adding 5 three times)
   - You're finding the total of equal groups

**Key differences:**
- **Addition** is about combining amounts: 3 + 5 means "3 plus 5"
- **Multiplication** is about repeated addition or scaling: 3 × 5 means "3 groups of 5" or "5 added to itself 3 times"
- **Result**: Addition usually gives a smaller result than multiplication (when both numbers are greater than 1)

**Example comparison:**
- Add: 3 + 5 = 8
- Multip